# **<center>Machine-Learning Prediction of AlGaN Bandgaps <br> Using Only DFT Structural Descriptors</center>**
<center><span style="font-size: 1.5em;">Bahram Abedi Ravan, Boshra Kiani Sadr and Aliasghar Shokri</span></center>
<center><span style="font-size: 1.2em;">17 May 2026</span></center>

## **✅✅✅ Data Loading**

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

#data = pd.read_excel('/content/drive/MyDrive/PAPERS/KianiMs/AlGaN_Data_040829.xlsx')
#data = pd.read_excel('AlGaN_Data_040830.xlsx')
data = pd.read_excel('AlGaN_DataSet.xlsx')
print(data.columns)
target = 'Bandgap'

## **✅✅✅ Encoding and Feature Selection**

In [ ]:
# Categorical to numerical
label_encoders = {}
categorical_cols = ['Phase', 'Method', 'Approximation', 'XC-Functional', 'Space Group', 'XC-Functional']
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

#X = data.drop(['Bandgap'], axis=1)
#X = data.drop(['Space Group', 'lc_a', 'lc_b', 'lc_c', 'Bandgap', 'Ref'], axis=1)
#X = data.drop(['Bandgap', 'Epsilon_xx', 'Epsilon_zz', 'B', 'Bˊ'], axis=1)
X = data.drop(['Bandgap', 'lc_a', 'lc_b', 'lc_c', 'Epsilon_xx', 'Epsilon_zz', 'B', 'BP', 'Ref'], axis=1)
#X = data.drop(['Bandgap', 'lc_a', 'lc_b', 'lc_c', 'Epsilon_xx', 'Epsilon_zz', 'B', 'BP', 'Ref', 'Space Group', 'Phase'], axis=1)

print(X.columns)
y = data[target]


In [ ]:
%%script echo skipping

# Feature selection - select top 6 features
selector = SelectKBest(score_func=f_regression, k=6)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]

print("Selected features for bandgap prediction:")
print(selected_features)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [ ]:

for col in X.columns:
  if col == 'Volume':
    X[col] = (X[col] - X[col].min()) / X[col].max()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=62)

#scaler = StandardScaler()
#X_train_scaled = scaler.fit_transform(X_train)
#X_test_scaled = scaler.transform(X_test)

## **✅✅✅ Evaluation of Different Models**

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso, LogisticRegression, LinearRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

models = {
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, subsample=0.5, random_state=42), 
    'XGBoost':              XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, eval_metric='rmse'),
    'SVR RBF':              SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1),  # renamed
    'Neural Network':       MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
    'Extra Trees':          ExtraTreesRegressor(n_estimators=200, random_state=42),
    'AdaBoost':             AdaBoostRegressor(n_estimators=200, random_state=42),
    'Ridge Regression':     Ridge(alpha=1.0),
    'Lasso Regression':     Lasso(alpha=0.001, max_iter=10000),
    'KNN Regressor':        KNeighborsRegressor(n_neighbors=5, weights='distance'),
    'LightGBM':             LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, num_leaves=25, feature_fraction=0.8, random_state=42, verbose=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=5, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    test_r2 = model.score(X_test, y_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'Test R2':    round(test_r2,        3),
        'CV Mean R2': round(cv_scores.mean(), 3),
    }
    print(f"{name}: Test R²={test_r2:.3f}, CV Mean R²={cv_scores.mean():.3f}")

#best_model = results['Gradient Boosting']['Model']    

## **✅✅✅ Sanity Check**

In [ ]:
%%script echo skipping
#@@@@@@@@@@@@@@@@@@	Sanity Check
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyRegressor

# Baseline model: predict mean Bandgap
dummy = DummyRegressor(strategy="mean")
scores = cross_val_score(dummy, X, y, cv=5, scoring="r2")
print(f"Baseline R²: {np.mean(scores):.3f}")

#@@@@@@@@@@@@@@@@@@	Cross Validation
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import ExtraTreesRegressor

model = ExtraTreesRegressor(n_estimators=200, random_state=62)
scores = cross_val_score(best_model, X, y, cv=5, scoring="r2")  # 5-fold CV
print(f"Cross-validated R²: {scores.mean():.3f} (±{scores.std():.3f})")

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

#@@@@@@@@@@@@@@@@@@	Scale features (critical for linear models)
X_scaled_local = StandardScaler().fit_transform(X)
scores = cross_val_score(LinearRegression(), X_scaled_local, y, cv=5, scoring="r2")
print(f"Linear Model R²: {scores.mean():.3f}")

In [ ]:
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.dummy import DummyRegressor

print("\n=== Sanity Check ===")

# K-fold cross validation with ExtraTrees
#cv_model = ExtraTreesRegressor(n_estimators=200, random_state=62)
#cv_scores = cross_val_score(cv_model, X, y, cv=5, scoring="r2")
#print(f"Cross-validated R²: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")

# Repeated K-Fold cross validation
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

# Baseline model (DummyRegressor)
dummy = DummyRegressor(strategy="mean")
baseline_scores = cross_val_score(dummy, X, y, cv=cv, scoring="r2")
print(f"Baseline R²: {baseline_scores.mean():.3f}")

# Linear model with scaling
#X_scaled_2 = StandardScaler().fit_transform(X)
linear_scores = cross_val_score(LinearRegression(), X, y, cv=cv, scoring="r2")
print(f"Linear Model R²: {linear_scores.mean():.3f}")

# ExtraTrees
# -----------------------------
cv_model = ExtraTreesRegressor(n_estimators=150, random_state=42, max_depth=10)
cv_scores2 = cross_val_score(cv_model, X, y, cv=cv, scoring="r2")
print(f"Repeated K-Fold R² (ExtraTrees): {cv_scores2.mean():.3f} (±{cv_scores2.std():.3f})")

# Gradient Boosting
# -----------------------------
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=62)
gbr_scores = cross_val_score(gbr, X, y, cv=cv, scoring="r2")
print(f"Repeated K-Fold R² (GradientBoosting): {gbr_scores.mean():.3f} (±{gbr_scores.std():.3f})")

# -----------------------------
# AdaBoost
# -----------------------------
abr = AdaBoostRegressor(n_estimators=100, learning_rate=0.8, random_state=42)
abr_scores = cross_val_score(abr, X, y, cv=cv, scoring="r2")
print(f"Repeated K-Fold R² (AdaBoost): {abr_scores.mean():.3f} (±{abr_scores.std():.3f})")

In [ ]:
# Gradient Boosting
gbr = GradientBoostingRegressor(n_estimators=150, learning_rate=0.2, max_depth=2, random_state=62)
gbr_scores = cross_val_score(gbr, X, y, cv=cv, scoring="r2")
mae_scores = -cross_val_score(gbr, X, y, cv=cv, scoring='neg_mean_absolute_error')
print(f"GradientBoosting (improved parameters):")
print(f"Repeated K-Fold R²: {gbr_scores.mean():.3f} (±{gbr_scores.std():.3f})")
print(f"Repeated K-Fold MAE: {mae_scores.mean():.3f} (±{mae_scores.std():.3f}) eV")

## **✅✅✅ GradientBoosting model using ONLY the "x" (composition)**

In [ ]:
X_xonly = X[['x']]

# !!! Check "gbr" in the "Repeated K-Fold" cell
scores = cross_val_score(gbr, X_xonly, y, cv=cv, scoring='r2')
mae_scores = -cross_val_score(gbr, X_xonly, y, cv=cv, scoring='neg_mean_absolute_error')

print(f"Repeated K-Fold R²: {scores.mean():.3f} (±{scores.std():.3f})")
print(f"Repeated K-Fold MAE: {mae_scores.mean():.3f} (±{mae_scores.std():.3f}) eV")

## **✅✅✅ End-Point Limits (x → 0 and x → 1)**

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

# !!! Check "gbr" in the "Repeated K-Fold" cell
y_pred_all = cross_val_predict(gbr, X, y, cv=5)

# Endpoint mask
mask_end = (X['x'] <= 0.125) | (X['x'] >= 0.875)

# Middle mask
mask_mid = (X['x'] > 0.125) & (X['x'] < 0.875)

# MAE calculations
mae_end = mean_absolute_error(y[mask_end], y_pred_all[mask_end])
mae_mid = mean_absolute_error(y[mask_mid], y_pred_all[mask_mid])

print(f"Endpoint MAE: {mae_end:.3f} eV")
print(f"Middle-region MAE: {mae_mid:.3f} eV")

## **📈✏️📊 PLOT: Learning Curve**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, title, X, y, cv=None,
                        train_sizes=np.linspace(0.1, 1.0, 10),
                        font_size=14, axes_lw=2.0):

    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, train_sizes=train_sizes, scoring='r2'
    )

    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std  = np.std(train_scores,  axis=1)
    test_scores_mean  = np.mean(test_scores,  axis=1)
    test_scores_std   = np.std(test_scores,   axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.fill_between(train_sizes,
                    train_scores_mean - train_scores_std,
                    train_scores_mean + train_scores_std,
                    alpha=0.1, color='r')
    ax.fill_between(train_sizes,
                    test_scores_mean - test_scores_std,
                    test_scores_mean + test_scores_std,
                    alpha=0.1, color='g')

    ax.plot(train_sizes, train_scores_mean, 'o-', color='r',
            linewidth=axes_lw, label='Training score')
    ax.plot(train_sizes, test_scores_mean,  'o-', color='g',
            linewidth=axes_lw, label='Cross-validation score')

    ax.set_title(title,               fontsize=font_size + 4)
    ax.set_xlabel('Training examples', fontsize=font_size + 2)
    ax.set_ylabel('Score (R²)',        fontsize=font_size + 2)
    ax.tick_params(labelsize=font_size, width=axes_lw)
    ax.legend(loc='best', fontsize=font_size)
    ax.grid()
    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)

    plt.tight_layout()
    plt.savefig("Figs/Learning_Curve.pdf", bbox_inches='tight', transparent=True)
    plt.show()

plot_learning_curve(gbr, "Learning Curve (Gradient Boosting)", X, y, cv=cv, font_size=14, axes_lw=2.0)

## **📈✏️📊 PLOT: SHAP analysis**

In [ ]:
import shap
import matplotlib.pyplot as plt

def shap_analysis(model, X, model_name="", font_size=14, axes_lw=2.0, max_display=15):

    explainer   = shap.Explainer(model)
    shap_values = explainer(X)

    plt.figure(figsize=(6, 4))   #max(4, max_display * 0.4)))
    shap.plots.beeswarm(shap_values, max_display=max_display, show=False)
    ax = plt.gca()
    plt.gcf().set_size_inches((10, 6))   #, max(10, max_display * 0.4))

#    ax.set_title(f'{model_name} — SHAP Feature Effects', fontsize=font_size + 4, pad=12)
    ax.set_xlabel('SHAP Value (impact on model output)', fontsize=font_size + 2)
    ax.tick_params(labelsize=font_size, width=axes_lw)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)

    fig = plt.gcf()
    cb_axes = [axes for axes in fig.axes if axes != ax]
    if cb_axes:
        cb = cb_axes[0]
        cb.tick_params(labelsize=font_size - 2)
        cb.set_ylabel('Feature value', fontsize=font_size)

    plt.tight_layout()
    plt.savefig("Figs/SHAP_Values.pdf", bbox_inches='tight')

gbr.fit(X_train, y_train)
shap_analysis(gbr, pd.DataFrame(X_train, columns=X.columns),
              model_name="Gradient Boosting",
              font_size=16, axes_lw=2.0, max_display=15)

## **📈✏️📊 PLOT: SHAP-Based Model Interpretability and Feature Analysis**

In [ ]:
def advanced_shap_analysis(model, X, y, font_size=14):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # ── 1. Actual vs Predicted + SHAP dependence on Al composition ──────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    y_pred = model.predict(X)
    r2 = np.corrcoef(y, y_pred)[0, 1] ** 2

    ax1.scatter(X['x'], y,       alpha=0.4, label='Actual',    color='#404040', s=20)
    ax1.scatter(X['x'], y_pred,  alpha=0.4, label='Predicted', color='#AAAAAA', s=20)
    ax1.set_xlabel('Al Composition (x)',  fontsize=font_size)
    ax1.set_ylabel('Bandgap (eV)',        fontsize=font_size)
    ax1.set_title('Actual vs Predicted by Al Composition', fontsize=font_size + 1)
    ax1.tick_params(labelsize=font_size)
    ax1.legend(fontsize=font_size)
    ax1.text(0.05, 0.92, f'R² = {r2:.3f}', transform=ax1.transAxes,
             fontsize=font_size, bbox=dict(facecolor='white', alpha=0.8))

    plt.sca(ax2)
    shap.dependence_plot('x', shap_values, X, interaction_index=None,
                         ax=ax2, show=False)
    ax2.set_xlabel('Al Composition (x)', fontsize=font_size)
    ax2.set_ylabel('SHAP Value',         fontsize=font_size)
    ax2.set_title('SHAP Dependence: Al Composition → Bandgap', fontsize=font_size + 1)
    ax2.tick_params(labelsize=font_size)

    plt.tight_layout()
    plt.show()

    # ── 2. Non-linear effects for categorical features ───────────────────────
    for feat in ['Space Group', 'Approximation']:
        fig, ax = plt.subplots(figsize=(8, 5))
        shap.dependence_plot(feat, shap_values, X,
                             interaction_index='auto',
                             ax=ax, show=False)
        ax.set_xlabel(feat,          fontsize=font_size)
        ax.set_ylabel('SHAP Value',  fontsize=font_size)
        ax.set_title(f'SHAP Dependence: {feat} (with auto interaction)', fontsize=font_size + 1)
        ax.tick_params(labelsize=font_size)
        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
        plt.tight_layout()
        plt.show()

    # ── 3. Model consistency: train vs test SHAP sums ───────────────────────
    shap_values_train = explainer.shap_values(X_train)
    shap_values_test  = explainer.shap_values(X_test)

    train_sum = shap_values_train.sum(axis=1) + explainer.expected_value
    test_sum  = shap_values_test.sum(axis=1)  + explainer.expected_value

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(train_sum, y_train, alpha=0.4, color='#404040', s=20, label='Train')
    ax.scatter(test_sum,  y_test,  alpha=0.4, color='#AAAAAA', s=20, label='Test')

    # Perfect prediction line
    all_vals = np.concatenate([train_sum, test_sum])
    lims = [all_vals.min(), all_vals.max()]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect fit')

    # Annotate train/test R²
    r2_train = np.corrcoef(train_sum, y_train)[0, 1] ** 2
    r2_test  = np.corrcoef(test_sum,  y_test)[0, 1]  ** 2
    ax.text(0.05, 0.92,
            f'Train R² = {r2_train:.3f}\nTest  R² = {r2_test:.3f}',
            transform=ax.transAxes, fontsize=font_size,
            bbox=dict(facecolor='white', alpha=0.8))

    ax.set_xlabel('SHAP Sum + Baseline (eV)', fontsize=font_size)
    ax.set_ylabel('Actual Bandgap (eV)',       fontsize=font_size)
    ax.set_title('Model Consistency: SHAP Prediction vs Actual', fontsize=font_size + 1)
    ax.tick_params(labelsize=font_size)
    ax.legend(fontsize=font_size)

    plt.tight_layout()
    plt.show()

advanced_shap_analysis(gbr, X, y, font_size=12)

## **📈✏️📊 PLOT: Performance Comparison -- Test $R^2$ and CV Mean $R^2$**

In [ ]:
%%script echo Skipping
import json
print(type(results))
print(json.dumps(results, indent=2, default=str))

In [ ]:
for name, metrics in results.items():
    print(name, metrics)             #  print(name, metrics.keys())
  #  break 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_model_comparison(results, font_size=11, top_n=6):
    model_names  = list(results.keys())
    test_r2      = [results[name]['Test R2']    for name in model_names]
    cv_mean_r2   = [results[name]['CV Mean R2'] for name in model_names]

    # Sort by CV Mean R² descending
    sorted_indices = np.argsort(cv_mean_r2)[::-1]
    model_names  = [model_names[i] for i in sorted_indices]
    test_r2      = [test_r2[i]     for i in sorted_indices]
    cv_mean_r2   = [cv_mean_r2[i]  for i in sorted_indices]

    # Keep only top N models
    model_names  = model_names[:top_n]
    test_r2      = test_r2[:top_n]
    cv_mean_r2   = cv_mean_r2[:top_n]

    n = len(model_names)
    y = np.arange(n)
    bar_height = 0.35

    fig, ax = plt.subplots(figsize=(10, max(6, n * 0.7)))

    bars_test = ax.barh(y + bar_height / 2, test_r2,    height=bar_height, color='skyblue', label='Test R²')                 # hatch='---', 
    bars_cv   = ax.barh(y - bar_height / 2, cv_mean_r2, height=bar_height, color='coral', label='CV Mean R²')                # hatch='...', 

    for bar in bars_test:
        width = bar.get_width()
        ax.text(width + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{width:.3f}', va='center', ha='left', fontsize=font_size - 1)

    for bar in bars_cv:
        width = bar.get_width()
        ax.text(width + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{width:.3f}', va='center', ha='left', fontsize=font_size - 1)

    ax.set_yticks(y)
    ax.set_yticklabels(model_names, fontsize=font_size)
    ax.set_xlabel('R² Score', fontsize=font_size)
#    ax.set_title(f'Performance Comparison', fontsize=font_size + 2)
    ax.tick_params(axis='both', labelsize=font_size)
    ax.set_xlim(0, 1.1)
    ax.grid(axis='x', linestyle='--', alpha=0.7)
    ax.legend(loc='upper right', fontsize=font_size-4)

    plt.tight_layout()
    plt.savefig("Figs/Model_Comparison.pdf", bbox_inches='tight')

plot_model_comparison(results, font_size=16, top_n=6)

## **📈✏️📊 PLOT: Feature Importance**

In [ ]:
def plot_feature_importances(X, y, model=gbr, font_size=14, axes_lw=2.0):
    model.fit(X, y)
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=X.columns)
    else:
        importances = pd.Series(np.abs(model.coef_), index=X.columns)

    importances = importances.sort_values(ascending=True)
    features    = importances.index.tolist()
    y_pos       = np.arange(len(features))

    fig, ax = plt.subplots(figsize=(8, max(6, len(features) * 0.4)))

    ax.barh(y_pos, importances.values, color='coral')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features, fontsize=font_size)
    ax.set_title('Model Feature Importances\n', fontsize=font_size + 4)
    ax.set_xlabel('Importance Score',           fontsize=font_size + 2)
    ax.set_ylabel('Feature',                    fontsize=font_size + 2)
    ax.tick_params(labelsize=font_size, width=axes_lw)
    ax.grid(axis='x', linestyle='--', alpha=0.8)
    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)

    plt.tight_layout()
    plt.savefig("Figs/Feature_Importances.pdf", bbox_inches='tight', transparent=True)
    plt.show()

    return importances

importances = plot_feature_importances(X, y)


## **📈✏️📊 PLOT: Feature Correlations**

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

def plot_feature_correlations(X, y, figsize=(9, 5)):
    """
    Calculate and plot feature correlations with target variable in descending order.

    Parameters:
    X (DataFrame): Features dataframe
    y (Series): Target variable
    figsize (tuple): Figure size (width, height)
    """
    # Calculate correlation coefficients
    correlations = X.corrwith(y).sort_values(ascending=False)

    # Create a DataFrame for plotting
    corr_df = pd.DataFrame({'Feature': correlations.index,
                           'Correlation': correlations.values})

    # Plot the correlations
    plt.figure(figsize=figsize)
    ax = sns.barplot(x='Correlation', y='Feature', data=corr_df, palette='viridis')

    # Add correlation values on the bars
    for i, (feature, corr) in enumerate(zip(corr_df['Feature'], corr_df['Correlation'])):
        ax.text(corr if corr > 0 else 0, i, f'{corr:.2f}',
                va='center', ha='left' if corr > 0 else 'right',
                color='black', fontsize=10)

    plt.title('Feature Correlations with Bandgap (eV)', fontsize=14)
    plt.xlabel('Correlation Coefficient', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    return correlations

feature_correlations = plot_feature_correlations(X, y)

In [ ]:
print(data.columns)
corr_data = X #.drop(['B', 'Bˊ', 'Epsilon_xx', 'Epsilon_zz'], axis=1)

# Calculate feature-feature correlations
plt.figure()
sns.heatmap(corr_data.corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.savefig("Figs/Correlation_Matrix.pdf", bbox_inches='tight')

## **📈✏️📊 PLOT: Data Inspection**

In [ ]:
plt.figure()
plt.scatter(data['x'], data['Bandgap'])
plt.xlabel('Al Composition (x)')
plt.ylabel('Bandgap (eV)')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create subplots: 2 rows, 3 columns
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
ax = axes.ravel()

ax1 = ax[0]
ax1.scatter(data.index, data.iloc[:, 7], label=data.columns[7])
ax1.scatter(data.index, data.iloc[:, 8], label=data.columns[8])
ax1.scatter(data.index, data.iloc[:, 9], label=data.columns[9])
ax1.set_xlabel('Index')
ax1.set_ylabel('LC values')
ax1.legend()
ax1.grid(True)

ax2 = ax[1]
ax2.scatter(data.iloc[:, 7], data[target], label=data.columns[7])
ax2.scatter(data.iloc[:, 8], data[target], label=data.columns[8])
ax2.scatter(data.iloc[:, 9], data[target], label=data.columns[9])
ax2.set_xlabel('LC values')
ax2.set_ylabel(target)
ax2.legend()
ax2.grid(True)

ax3 = ax[2]
ax3.scatter(data.iloc[:, 2], data[target], label=data.columns[2])
ax3.set_xlabel(data.columns[2])
ax3.set_ylabel(target)
ax3.legend()
ax3.grid(True)

ax4 = ax[3]
ax4.scatter(data.iloc[:, 3], data[target], label=data.columns[3])
ax4.set_xlabel(data.columns[3])
ax4.set_ylabel(target)
ax4.legend()
ax4.grid(True)

ax5 = ax[4]
ax5.scatter(data.iloc[:, 4], data[target], label=data.columns[4])
ax5.set_xlabel(data.columns[4])
ax5.set_ylabel(target)
ax5.legend()
ax5.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))

plt.subplot(221)
plt.scatter(data['x'], data['Bandgap'])
plt.xlabel('Composition (x)')
plt.ylabel('Bandgap (eV)')

plt.subplot(222)
plt.scatter(data['Epsilon_xx'], data['Bandgap'])
plt.xlabel('$\epsilon_{xx}$')
#plt.ylabel('Bandgap (eV)')

plt.subplot(223)
plt.scatter(data['B'], data['Bandgap'])
plt.xlabel('B')
plt.ylabel('Bandgap (eV)')

plt.subplot(224)
plt.scatter(data['BP'], data['Bandgap'])
plt.xlabel('Bˊ')
#plt.ylabel('Bandgap (eV)')

plt.show()

## **📈✏️📊 PLOT: Bandgap Prediction - Single Split**

In [ ]:
pred_train = gbr.predict(X_train)
pred_test  = gbr.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)
r2_test = r2_score(y_test, pred_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

ax1.scatter(y_train, pred_train, color='red', alpha=0.4)
ax1.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
ax1.set_xlabel('Actual Bandgap (eV)')
ax1.set_ylabel('Predicted Bandgap (eV)')
ax1.set_title(f'MSE = {mse_train: .3f} & $ R^2$ = {r2_train: .3f}')
ax1.text(2, 6, '(a) Train')

ax2.scatter(y_test, pred_test, color='red', alpha=0.4)
ax2.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
ax2.set_xlabel('Actual Bandgap (eV)')
#ax2.set_ylabel('Predicted Bandgap (eV)')
ax2.set_title(f'MSE = {mse_test: .3f} & $ R^2$ = {r2_test: .3f}')
ax2.text(2, 6, '(b) Test')

plt.savefig("Figs/Prediction_Diagonal_Train_Test.pdf", bbox_inches='tight')

In [ ]:
pred_train = gbr.predict(X_train)
pred_test  = gbr.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
mse_test  = mean_squared_error(y_test,  pred_test)
r2_train  = r2_score(y_train, pred_train)
r2_test   = r2_score(y_test,  pred_test)

font_size = 18
axes_lw   = 2.5

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_train, pred_train, color='red', marker='s', alpha=1, s=40, label='Train')
ax.scatter(y_test,  pred_test,  color='blue', marker='o', alpha=1, s=40, label='Test')
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=1.5)

ax.set_xlabel('Actual Bandgap (eV)',        fontsize=font_size)
ax.set_ylabel('Predicted Bandgap (eV)',     fontsize=font_size)
#ax.set_title('Actual vs Predicted Bandgap (Train & Test)', fontsize=font_size + 2)
ax.tick_params(axis='both', labelsize=font_size)

# Axes border thickness
for spine in ax.spines.values():
    spine.set_linewidth(axes_lw)
ax.tick_params(width=axes_lw)

ax.text(0.05, 0.83,
        f'Train — MSE: {mse_train:.3f}  R²: {r2_train:.3f}\n'
        f'Test  — MSE: {mse_test:.3f}  R²: {r2_test:.3f}',
        transform=ax.transAxes, fontsize=font_size-4,
        verticalalignment='top',
        bbox=dict(facecolor='white', alpha=0.8))

ax.legend(fontsize=font_size-2)
plt.tight_layout()
plt.savefig("Figs/Prediction_Diagonal.pdf", bbox_inches='tight')

## **📈✏️📊 PLOT: Bandgap Prediction - Repeated CV**

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_true_all = []
y_pred_all = []

cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

for train_idx, test_idx in cv.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = clone(gbr)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    y_true_all.extend(y_test)
    y_pred_all.extend(preds)

mse_cv = mean_squared_error(y_true_all, y_pred_all)
mae_cv = mean_absolute_error(y_true_all, y_pred_all)
r2_cv  = r2_score(y_true_all, y_pred_all)

print(f"MSE  = {mse_cv:.3f}")
print(f"MAE  = {mae_cv:.3f}")
print(f"R²   = {r2_cv:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse_cv = mean_squared_error(y_true_all, y_pred_all)
mae_cv = mean_absolute_error(y_true_all, y_pred_all)
r2_cv  = r2_score(y_true_all, y_pred_all)

# print(f"Repeated CV MSE : {mse_cv:.3f}")
# print(f"Repeated CV MAE : {mae_cv:.3f}")
# print(f"Repeated CV R²  : {r2_cv:.3f}")

font_size = 18
axes_lw   = 2.5

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true_all, y_pred_all, color='blue', marker='o', alpha=0.75, s=40, edgecolors='black', linewidths=0.4, label='Repeated CV Predictions')

min_val = min(min(y_true_all), min(y_pred_all))
max_val = max(max(y_true_all), max(y_pred_all))

ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=1.8)
ax.set_xlabel('Actual Bandgap (eV)', fontsize=font_size)
ax.set_ylabel('Predicted Bandgap (eV)', fontsize=font_size)
ax.tick_params(axis='both', labelsize=font_size, width=axes_lw)

for spine in ax.spines.values():
    spine.set_linewidth(axes_lw)

ax.text(
    0.10,
    0.88,
#    f'Repeated 5-Fold CV\n (10 Repeats)\n\n'
    f'MSE  = {mse_cv:.3f}\n'
    f'MAE  = {mae_cv:.3f} eV\n'
    f'R²     = {r2_cv:.3f}',
    transform=ax.transAxes,
    fontsize=font_size - 4,
    verticalalignment='top',
    bbox=dict(
        facecolor='white',
        edgecolor='black',
        alpha=0.85)  )

ax.legend(fontsize=font_size - 2)
plt.tight_layout()
plt.savefig("Figs/Prediction_Diagonal_RepeatedCV.pdf", bbox_inches='tight')

## **📈✏️📊 PLOT: Error Bars - Single Split**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

font_size = 14
axes_lw   = 2.5

# Ensure these are 1D arrays
y_train    = np.asarray(y_train).flatten()
pred_train = np.asarray(pred_train).flatten()
y_test     = np.asarray(y_test).flatten()
pred_test  = np.asarray(pred_test).flatten()

n_train = len(y_train)
n_test  = len(y_test)
width   = 0.35

x_train = np.arange(n_train)
x_test  = np.arange(n_test)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ── Training ─────────────────────────────────────────────────────────────────
ax1.bar(x_train - width/2, y_train,    width, label='Actual',    color='red')
ax1.bar(x_train + width/2, pred_train, width, label='Predicted', color='blue')
ax1.set_xlabel('Sample #',                          fontsize=font_size)
ax1.set_ylabel('Bandgap (eV)',                      fontsize=font_size)
ax1.set_title('Training Data:\nActual & Predicted Bandgaps', fontsize=font_size + 1)
ax1.tick_params(axis='both', labelsize=font_size, width=axes_lw)
ax1.legend(loc='upper right', fontsize=font_size-2)
ax1.text(2, 6, '(a)', fontsize=font_size)
for spine in ax1.spines.values():
    spine.set_linewidth(axes_lw)

# ── Testing ──────────────────────────────────────────────────────────────────
ax2.bar(x_test - width/2, y_test,    width, label='Actual',    color='red')
ax2.bar(x_test + width/2, pred_test, width, label='Predicted', color='blue')
ax2.set_xlabel('Sample #',                         fontsize=font_size)
ax2.set_ylabel('Bandgap (eV)',                     fontsize=font_size)
ax2.set_title('Testing Data:\nActual & Predicted Bandgaps', fontsize=font_size + 1)
ax2.tick_params(axis='both', labelsize=font_size, width=axes_lw)
ax2.legend(loc='upper right', fontsize=font_size-2)
ax2.text(2, 6, '(b)', fontsize=font_size)
for spine in ax2.spines.values():
    spine.set_linewidth(axes_lw)

plt.tight_layout()
plt.savefig("Figs/Prediction_Comparison.pdf", bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create figure and axes
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
ax1, ax2, ax3, ax4 = axes.ravel()

# Your data
x_train = np.arange(n_train)
x_test = np.arange(n_test)

y_train_a_err = np.asarray(np.abs(y_train - pred_train)).flatten()
y_test_a_err = np.asarray(np.abs(y_test - pred_test)).flatten()

y_train_r_err = np.asarray(np.abs(y_train - pred_train) / y_train * 100).flatten()
y_test_r_err = np.asarray(np.abs(y_test - pred_test) / y_test * 100).flatten()


font_size = 16
plt.rcParams['font.size'] = 16
plt.rcParams['font.family'] = 'Arial' 
plt.rcParams['axes.linewidth'] = 2.5

# Absolute errors
ax1.bar(x_train, y_train_a_err)
ax1.set_ylabel('Absolute Error (eV)', fontsize=font_size)
ax1.set_title('Training Data', fontsize=font_size)
# Automatic positioning for (a) - top left corner based on axes limits
ax1.text(0.02, 0.85, '(a)', transform=ax1.transAxes, fontsize=font_size, fontweight='bold')
# Add box with min and max values
ax1.text(0.98, 0.95, f'min: {np.min(y_train_a_err):.4f}\nmax: {np.max(y_train_a_err):.4f}', 
         transform=ax1.transAxes, fontsize=font_size-2, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

ax2.bar(x_test, y_test_a_err)
ax2.set_title('Testing Data', fontsize=font_size)
ax2.text(0.02, 0.85, '(b)', transform=ax2.transAxes, fontsize=font_size, fontweight='bold')
ax2.text(0.98, 0.95, f'min: {np.min(y_test_a_err):.4f}\nmax: {np.max(y_test_a_err):.4f}', 
         transform=ax2.transAxes, fontsize=font_size-2, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

# Relative errors
ax3.bar(x_train, y_train_r_err)
ax3.set_xlabel('Sample #', fontsize=font_size)
ax3.set_ylabel('Relative Error (%)', fontsize=font_size)
ax3.text(0.02, 0.85, '(c)', transform=ax3.transAxes, fontsize=font_size, fontweight='bold')
ax3.text(0.98, 0.95, f'min: {np.min(y_train_r_err):.2f}%\nmax: {np.max(y_train_r_err):.2f}%', 
         transform=ax3.transAxes, fontsize=font_size-2, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

ax4.bar(x_test, y_test_r_err)
ax4.set_xlabel('Sample #', fontsize=font_size)
ax4.text(0.02, 0.85, '(d)', transform=ax4.transAxes, fontsize=font_size, fontweight='bold')
ax4.text(0.98, 0.95, f'min: {np.min(y_test_r_err):.2f}%\nmax: {np.max(y_test_r_err):.2f}%', 
         transform=ax4.transAxes, fontsize=font_size-2, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

plt.tight_layout()
plt.savefig("Figs/Errors.pdf", bbox_inches='tight')

In [ ]:
def plot_residuals_combined(model, X_train, y_train, X_test, y_test, 
                             model_name='Gradient Boosting', font_size=16, axes_lw=2.5):

    pred_train = model.predict(X_train)
    pred_test  = model.predict(X_test)
    res_train  = np.asarray(y_train).flatten() - pred_train
    res_test   = np.asarray(y_test).flatten()  - pred_test

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    def style_ax(ax):
        ax.tick_params(labelsize=font_size, width=axes_lw)
        for spine in ax.spines.values():
            spine.set_linewidth(axes_lw)
        ax.grid(linestyle='--', alpha=0.5)

    # ── (a) Residuals vs Predicted ───────────────────────────────────────────
    ax = axes[0, 0]
    ax.scatter(pred_train, res_train, color='red',  marker='s', alpha=0.5, s=40, label='Train')
    ax.scatter(pred_test,  res_test,  color='blue', marker='o', alpha=0.7, s=40, label='Test')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
    ax.set_xlabel('Predicted Bandgap (eV)', fontsize=font_size)
    ax.set_ylabel('Residuals (eV)',         fontsize=font_size)
    ax.set_title('(a) Residuals vs Predicted', fontsize=font_size + 1)
    ax.text(0.02, 0.95,
            f'Train  mean: {res_train.mean():.3f}  std: {res_train.std():.3f}\n'
            f'Test   mean: {res_test.mean():.3f}  std: {res_test.std():.3f}',
            transform=ax.transAxes, fontsize=font_size - 1,
            verticalalignment='top', bbox=dict(facecolor='white', alpha=0.8))
    ax.legend(fontsize=font_size)
    style_ax(ax)

    # ── (b) Residuals vs Actual ──────────────────────────────────────────────
    ax = axes[0, 1]
    ax.scatter(np.asarray(y_train).flatten(), res_train, color='red',  marker='s', alpha=0.5, s=40, label='Train')
    ax.scatter(np.asarray(y_test).flatten(),  res_test,  color='blue', marker='o', alpha=0.7, s=40, label='Test')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
    ax.set_xlabel('Actual Bandgap (eV)', fontsize=font_size)
    ax.set_ylabel('Residuals (eV)',      fontsize=font_size)
    ax.set_title('(b) Residuals vs Actual', fontsize=font_size + 1)
    ax.legend(fontsize=font_size)
    style_ax(ax)

    # ── (c) Residual distribution ────────────────────────────────────────────
    ax = axes[1, 0]
    bins = np.linspace(min(res_train.min(), res_test.min()),
                       max(res_train.max(), res_test.max()), 25)
    ax.hist(res_train, bins=bins, color='red',  alpha=0.6, label='Train')
    ax.hist(res_test,  bins=bins, color='blue', alpha=0.8, label='Test')
    ax.axvline(0, color='red', linestyle='--', linewidth=1.2)
    ax.set_xlabel('Residuals (eV)', fontsize=font_size)
    ax.set_ylabel('Count',          fontsize=font_size)
    ax.set_title('(c) Residual Distribution', fontsize=font_size + 1)
    ax.legend(fontsize=font_size)
    style_ax(ax)

    # ── (d) Absolute error vs Predicted (highlights large errors) ────────────
    ax = axes[1, 1]
    ax.scatter(pred_train, np.abs(res_train), color='red',  marker='s', alpha=0.5, s=40, label='Train')
    ax.scatter(pred_test,  np.abs(res_test),  color='blue', marker='o', alpha=0.7, s=40, label='Test')
    ax.axhline(np.abs(res_test).mean(), color='red', linestyle='--',
               linewidth=1.2, label=f'Test mean abs error: {np.abs(res_test).mean():.3f}')
    ax.set_xlabel('Predicted Bandgap (eV)', fontsize=font_size)
    ax.set_ylabel('Absolute Error (eV)',    fontsize=font_size)
    ax.set_title('(d) Absolute Error vs Predicted', fontsize=font_size + 1)
    ax.legend(fontsize=font_size)
    style_ax(ax)

    fig.suptitle(f'Residual Analysis: {model_name}', fontsize=font_size + 3, y=1.01)
    plt.tight_layout()
    plt.savefig("Figs/Residual_Analysis.pdf", bbox_inches='tight')

plot_residuals_combined(gbr, X_train, y_train, X_test, y_test)

## **📈✏️📊 PLOT: Error Bars - Repeated CV**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

y_true_all = np.asarray(y_true_all).flatten()
y_pred_all = np.asarray(y_pred_all).flatten()

abs_err = np.abs(y_true_all - y_pred_all)
rel_err = np.abs((y_true_all - y_pred_all) / (y_true_all + 1e-12)) * 100

x_all = np.arange(len(y_true_all))

font_size = 16

plt.rcParams['font.size'] = font_size
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = 2.5

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
ax1, ax2 = axes.ravel()

ax1.bar(x_all, abs_err, width=1.0)
ax1.set_ylabel('Absolute Error (eV)', fontsize=font_size)
ax1.set_title('Repeated 5-Fold Cross-Validation Errors', fontsize=font_size)
ax1.text(0.02, 0.88, '(a)', transform=ax1.transAxes, fontsize=font_size, fontweight='bold')

ax1.text(
    0.98, 0.95,
    f'Min: {np.min(abs_err):.4f} eV\nMax: {np.max(abs_err):.4f} eV\nMean: {np.mean(abs_err):.4f} eV',
    transform=ax1.transAxes,
    fontsize=font_size - 2,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray')
)

ax2.bar(x_all, rel_err, width=1.0)
ax2.set_xlabel('Prediction Sample #', fontsize=font_size)
ax2.set_ylabel('Relative Error (%)', fontsize=font_size)
ax2.text(0.02, 0.88, '(b)', transform=ax2.transAxes, fontsize=font_size, fontweight='bold')

ax2.text(
    0.98, 0.95,
    f'Min: {np.min(rel_err):.2f}%\nMax: {np.max(rel_err):.2f}%\nMean: {np.mean(rel_err):.2f}%',
    transform=ax2.transAxes,
    fontsize=font_size - 2,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, edgecolor='gray')
)

for ax in [ax1, ax2]:
    ax.tick_params(axis='both', labelsize=font_size, width=2.5)
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)

plt.tight_layout()
plt.savefig("Figs/Errors_RepeatedCV.pdf", bbox_inches='tight')

In [ ]:
print(f'Min_abs_err: {np.min(abs_err):.2f}%   &   Max_abs_err: {np.max(abs_err):.2f}%   &   Mean_abs_err: {np.mean(abs_err):.2f}%')
print(f'Min_rel_err: {np.min(rel_err):.2f}%   &   Max_rel_err: {np.max(rel_err):.2f}%   &   Mean_rel_err: {np.mean(rel_err):.2f}%')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y_true_all = np.asarray(y_true_all).flatten()
y_pred_all = np.asarray(y_pred_all).flatten()

residuals = y_true_all - y_pred_all
rel_error = ((y_pred_all - y_true_all) / (y_true_all + 1e-12)) * 100

font_size = 16
axes_lw = 2.5

plt.rcParams['font.size'] = font_size
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.linewidth'] = axes_lw

# =========================================================
# 1) Residual Distribution Histogram
# =========================================================
fig1, ax1 = plt.subplots(figsize=(8,6))

ax1.hist(residuals, bins=40)

ax1.axvline(0, linestyle='--', linewidth=2)

ax1.set_xlabel('Residual Error (eV)', fontsize=font_size)
ax1.set_ylabel('Count', fontsize=font_size)
ax1.set_title('Residual Error Distribution (Repeated CV)', fontsize=font_size)

mean_res = np.mean(residuals)
std_res  = np.std(residuals)

ax1.text(
    0.97, 0.95,
    f'Mean = {mean_res:.3f} eV\nStd = {std_res:.3f} eV',
    transform=ax1.transAxes,
    fontsize=font_size-2,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray')
)

ax1.tick_params(axis='both', labelsize=font_size, width=axes_lw)

for spine in ax1.spines.values():
    spine.set_linewidth(axes_lw)

plt.tight_layout()
plt.savefig("Figs/Residual_Histogram_RepeatedCV.pdf", bbox_inches='tight')

# =========================================================
# 2) Relative Error Distribution Histogram
# =========================================================
fig2, ax2 = plt.subplots(figsize=(8,6))

ax2.hist(rel_error, bins=50)

ax2.axvline(0, linestyle='--', linewidth=2)

ax2.set_xlabel('Relative Error (%)', fontsize=font_size)
ax2.set_ylabel('Count', fontsize=font_size)
ax2.set_title('Relative Error Distribution (Repeated CV)', fontsize=font_size)

mean_rel = np.mean(rel_error)
std_rel  = np.std(rel_error)

ax2.text(
    0.97, 0.95,
    f'Mean = {mean_rel:.2f}%\nStd = {std_rel:.2f}%',
    transform=ax2.transAxes,
    fontsize=font_size-2,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray')
)

ax2.tick_params(axis='both', labelsize=font_size, width=axes_lw)

for spine in ax2.spines.values():
    spine.set_linewidth(axes_lw)

plt.tight_layout()
plt.savefig("Figs/Relative_Error_Histogram_RepeatedCV.pdf", bbox_inches='tight')

plt.show()

## **📈✏️📊 PLOT: Correlations**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

def plot_feature_correlations(X, y, figsize=(10, 5)):
    correlations = X.corrwith(y).sort_values(ascending=False)
    corr_df = pd.DataFrame({'Feature': correlations.index,
                           'Correlation': correlations.values})

    plt.figure(figsize=figsize)
    ax = sns.barplot(x='Correlation', y='Feature', data=corr_df, palette='viridis')
    for i, (feature, corr) in enumerate(zip(corr_df['Feature'], corr_df['Correlation'])):
        ax.text(corr if corr > 0 else 0, i, f'{corr:.2f}',
                va='center', ha='left' if corr > 0 else 'right',
                color='black', fontsize=12)

    plt.title('Feature Correlations with Bandgap (eV)', fontsize=18)
    plt.xlabel('Correlation Coefficient', fontsize=16)
    plt.ylabel('Features', fontsize=16)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig("Figs/Feature correlations with bandgap.pdf", bbox_inches='tight')

    return correlations

feature_correlations = plot_feature_correlations(X, y)
print("\nFeature correlations with bandgap (sorted by absolute value):")
print(feature_correlations.abs().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.savefig("Figs/Feature-Feature Correlation Matrix.pdf", bbox_inches='tight')

In [ ]:
def analyze_features(X, y, model=gbr, font_size=18, axes_lw=2.5):

    y_series     = pd.Series(np.asarray(y).flatten(), name='target')
    correlations = X.corrwith(y_series).sort_values(ascending=False)

    model.fit(X, y)
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=X.columns)
    else:
        importances = pd.Series(np.abs(model.coef_), index=X.columns)

    feature_analysis = pd.DataFrame({
        'Correlation': correlations,
        'Importance':  importances
    }).sort_values('Importance', ascending=True)

    features = feature_analysis.index.tolist()
    y_pos    = np.arange(len(features))

    font_size = 14
    axes_lw = 2.0
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(6, len(features) * 0.4)),
                                   sharey=False)

    ax1.barh(y_pos, feature_analysis['Correlation'].values, color='red')
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(features, fontsize=font_size)
    ax1.set_title('Feature Correlations with Target \n', fontsize=font_size + 4)
    ax1.set_xlabel('Pearson Correlation Coefficient', fontsize=font_size+2)
    ax1.set_ylabel('Feature',                         fontsize=font_size+2)
    ax1.axvline(0, color='black', linewidth=0.8)
    ax1.tick_params(labelsize=font_size, width=axes_lw)
    ax1.grid(axis='x', linestyle='--', alpha=0.7)
    for spine in ax1.spines.values():
        spine.set_linewidth(axes_lw)

    ax2.barh(y_pos, feature_analysis['Importance'].values, color='blue')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(features, fontsize=font_size)
    ax2.set_title('Model Feature Importances \n', fontsize=font_size + 4)
    ax2.set_xlabel('Importance Score',         fontsize=font_size+2)
    ax2.tick_params(labelsize=font_size, width=axes_lw)
    ax2.grid(axis='x', linestyle='--', alpha=0.7)
    for spine in ax2.spines.values():
        spine.set_linewidth(axes_lw)

    plt.tight_layout()
    plt.savefig("Figs/Feature_Analysis.pdf", dpi=300, bbox_inches='tight', transparent=True)
    plt.show()

    return feature_analysis

feature_analysis = analyze_features(X, y)

In [ ]:
def plot_feature_correlations(X, y, font_size=14, axes_lw=2.0):
    y_series     = pd.Series(np.asarray(y).flatten(), name='target')
    correlations = X.corrwith(y_series).sort_values(ascending=True)

    features = correlations.index.tolist()
    y_pos    = np.arange(len(features))

    fig, ax = plt.subplots(figsize=(10, max(6, len(features) * 0.4)))

    ax.barh(y_pos, correlations.values, color='red')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features, fontsize=font_size)
    ax.set_title('Feature Correlations with Target\n', fontsize=font_size + 4)
    ax.set_xlabel('Pearson Correlation Coefficient',   fontsize=font_size + 2)
    ax.set_ylabel('Feature',                           fontsize=font_size + 2)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.tick_params(labelsize=font_size, width=axes_lw)
    ax.grid(axis='x', linestyle='--', alpha=0.7)
    for spine in ax.spines.values():
        spine.set_linewidth(axes_lw)

    plt.tight_layout()
    plt.savefig("Figs/Feature_Correlations.pdf", bbox_inches='tight', transparent=True)
    plt.show()

    return correlations

correlations = plot_feature_correlations(X, y)

## **📈✏️📊 PLOT: Feature Importance & Correlation Matrix**

In [ ]:
def plot_importance_and_correlation(X, y, model=gbr, font_size=16, axes_lw=2.5):

    # ── Data prep ────────────────────────────────────────────────────────────
    model.fit(X, y)
    if hasattr(model, 'feature_importances_'):
        importances = pd.Series(model.feature_importances_, index=X.columns)
    else:
        importances = pd.Series(np.abs(model.coef_), index=X.columns)
    importances = importances.sort_values(ascending=True)
    features    = importances.index.tolist()
    y_pos       = np.arange(len(features))

    # Add bandgap target to correlation matrix
    X_with_target = X.copy()
    X_with_target['Bandgap (eV)'] = np.asarray(y).flatten()
    corr_matrix = X_with_target.corr()

    # ── Layout ───────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(7, len(features) * 0.5)))

    # ── (a) Feature importances ──────────────────────────────────────────────
    ax1.barh(y_pos, importances.values, color='coral')
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(features,                       fontsize=font_size)
    ax1.set_title('(a) Model Feature Importances\n',    fontsize=font_size + 4)
    ax1.set_xlabel('Importance Score',                  fontsize=font_size + 2)
    ax1.set_ylabel('Feature',                           fontsize=font_size + 2)
    ax1.tick_params(labelsize=font_size, width=axes_lw)
    ax1.grid(axis='x', linestyle='--', alpha=0.8)
    for spine in ax1.spines.values():
        spine.set_linewidth(axes_lw)

    # ── (b) Feature-feature + target correlation heatmap ────────────────────
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0,
                ax=ax2, annot_kws={'size': font_size - 2},
                linewidths=0.5, linecolor='grey')
    ax2.set_title('(b) Feature & Target Correlation Matrix\n', fontsize=font_size + 4)
    ax2.tick_params(labelsize=font_size, width=axes_lw)
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=font_size - 1)
    ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0,              fontsize=font_size - 1)

    # Highlight the Bandgap column/row with a thicker border
    for _, spine in ax2.spines.items():
        spine.set_linewidth(axes_lw)

    plt.tight_layout()
    plt.savefig("Figs/Importance_and_Correlation.pdf", bbox_inches='tight', transparent=True)
    plt.show()

    return importances, corr_matrix

importances, corr_matrix = plot_importance_and_correlation(X, y)